# Lab 4.2 · Activity 4.1 and Lab 4.1 — Again, in Gephi
**Chapter 4 · Social Network Analytics I** · do this lab **after Chapter 6 (Gephi workshop)**

**Goal:** compute the same measures **in Gephi** — first for the karate club (Activity 4.1), then for the EV mention network (Lab 4.1) — and **check them against Python**.
**Data:** this notebook writes two GEXF files to `outputs/` with the Python results stored as node attributes, so you can compare both tools side by side in Gephi’s **Data Laboratory**.

📖 Interactive study guide: [Chapter 4](https://mehrdadjalali-ai.github.io/SocialMediaAnayltics/interactive/Chapter4_Networks_Influence_Communities.html) · Gephi steps: [Chapter 6 study guide](https://mehrdadjalali-ai.github.io/SocialMediaAnayltics/interactive/Chapter6_Gephi_Project.html)

| File written to `outputs/` | Open it in Gephi for |
|---|---|
| `karate_lab42.gexf` | Part A — every measure from Activity 4.1 |
| `ev_lab42.gexf` | Part B — the EV mention network from Lab 4.1 |

In [1]:
from pathlib import Path
import warnings; warnings.filterwarnings("ignore")
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA, OUTD = ROOT / "data", ROOT / "outputs"
OUTD.mkdir(exist_ok=True)
SEED = 42
import pandas as pd, networkx as nx, matplotlib.pyplot as plt

## Step 0 · Write the two GEXF files (Python reference values inside)
Every node gets the Python results as attributes with the ending `_py` — Gephi adds its own columns next to them.

In [2]:
def add_python_measures(G, directed=None):
    """Store degree, betweenness, closeness, PageRank and Louvain community as node attributes."""
    U = G.to_undirected()
    D = directed if directed is not None else G
    comms = nx.community.louvain_communities(U, seed=SEED)
    vals = {
        "degree_py": dict(U.degree()),
        "betweenness_py": nx.betweenness_centrality(U),
        "closeness_py": nx.closeness_centrality(U),
        "pagerank_py": nx.pagerank(D, weight="weight"),
        "community_py": {n: i for i, c in enumerate(comms) for n in c},
    }
    for name, d in vals.items():
        nx.set_node_attributes(G, {k: (round(v, 4) if isinstance(v, float) else v) for k, v in d.items()}, name)
    return U, comms

K = nx.karate_club_graph()
UK, ck = add_python_measures(K)
nx.write_gexf(K, OUTD / "karate_lab42.gexf")

m = pd.read_csv(DATA / "ev_mentions.csv"); users = pd.read_csv(DATA / "ev_users.csv")
E = nx.from_pandas_edgelist(m, "source", "target", ["weight"], create_using=nx.DiGraph)
E.add_nodes_from(users.user_id)
nx.set_node_attributes(E, users.set_index("user_id")[["lang", "group"]].to_dict("index"))
UE = E.to_undirected()
nx.set_node_attributes(UE, dict(E.nodes(data=True)))
_, ce = add_python_measures(UE, directed=E)
nx.write_gexf(UE, OUTD / "ev_lab42.gexf")              # undirected, like the Python measures
print("saved:", OUTD / "karate_lab42.gexf", "and", OUTD / "ev_lab42.gexf")

saved: /home/claude/socialmediaanayltics/outputs/karate_lab42.gexf and /home/claude/socialmediaanayltics/outputs/ev_lab42.gexf


### Python reference values — your Gephi results should match these

In [3]:
def summary(U, comms):
    out = {"nodes": U.number_of_nodes(), "edges": U.number_of_edges(),
            "average degree": round(2 * U.number_of_edges() / U.number_of_nodes(), 2),
            "density": round(nx.density(U), 3), "diameter": nx.diameter(U),
            "avg. path length": round(nx.average_shortest_path_length(U), 2),
            "avg. clustering": round(nx.average_clustering(U), 3),
            "communities (Louvain)": len(comms), "modularity": round(nx.community.modularity(U, comms), 3)}
    return {k: str(v) for k, v in out.items()}
pd.DataFrame({"Karate club (Part A)": summary(UK, ck), "EV network (Part B)": summary(UE, ce)})

,Karate club (Part A),EV network (Part B)
nodes,34,260
edges,78,1188
average degree,4.59,9.14
density,0.139,0.035
diameter,5,5
avg. path length,2.41,2.77
avg. clustering,0.571,0.092
communities (Louvain),4,5
modularity,0.427,0.443


## Part A · The karate club in Gephi
Open `outputs/karate_lab42.gexf` (**File → Open**, graph type **Undirected**). Then follow the table — every row is one section of Activity 4.1.

| # | Activity 4.1 section | In Gephi |
|---|---|---|
| A1 | Look inside | **Data Laboratory** → Nodes / Edges tables: `club`, `weight` and the `_py` columns |
| A2 | Draw | **Layout → ForceAtlas 2** (Scaling 10, *Prevent Overlap* on) → Run, then Stop |
| A3 | Whole network | **Statistics**: *Average Degree*, *Graph Density*, *Network Diameter* (diameter + path length), *Avg. Clustering Coefficient*, *Connected Components* |
| A4 | Traversal (BFS) | **Filters → Topology → Ego Network**: node 0, depth 1, then 2, then 3 — each depth is one BFS level |
| A5 | Centralities | *Network Diameter* also computes **betweenness** and **closeness** · run **PageRank** (p = 0.85) |
| A6 | Size by centrality | **Appearance → Nodes → Size → Ranking → Betweenness Centrality** (min 10, max 50) |
| A7 | Communities | **Statistics → Modularity** (resolution 1.0, *Use weights* on) → **Appearance → Color → Partition → Modularity Class** |
| A8 | Homophily | **Appearance → Color → Partition → club** — compare with the Modularity colours |
| A9 | Bridges and brokers | **Filters → Attributes → Partition → club**: show one group, then the other — which nodes keep ties across? |
| A10 | Export | **Preview** → *Show labels* → Export **SVG/PNG** |

## Part B · The EV mention network in Gephi
Open `outputs/ev_lab42.gexf` (**Undirected**, like the Python measures). Repeat **A2–A7**, then:

| # | Lab 4.1 challenge | In Gephi |
|---|---|---|
| B1 | Influence | **Data Laboratory**: sort by *Betweenness Centrality*, then by *PageRank* — top 5 each |
| B2 | Communities | **Modularity** at resolution **0.5, 1.0 and 1.5** — note number of communities and Q each time |
| B3 | Name the communities | Colour by **Modularity Class**, then by **group** and **lang** — which colour matches which group? |
| B4 | Bridge users | **Filters → Topology → Ego Network** on `u196` (depth 1): how many different community colours do its neighbours have? |
| B5 | Figure | Size = PageRank, colour = Modularity Class, labels only for the top 5 → export SVG |

## Challenge 1 · Do Gephi and Python agree?
Fill in the comparison table for **both** networks. Use the Python values above.

| Measure | Karate · Python | Karate · Gephi | EV · Python | EV · Gephi |
|---|---|---|---|---|
| average degree | | | | |
| density | | | | |
| diameter · avg. path length | | | | |
| avg. clustering | | | | |
| top 3 betweenness | | | | |
| top 3 PageRank | | | | |
| communities · modularity | | | | |

> 💡 **Hint:** Gephi’s betweenness is **not normalised** by default (tick *Normalize centralities in [0,1]* in Network Diameter to match Python).

**Your notes (Challenge 1):** _which values match exactly, which differ slightly?_

## Challenge 2 · Two figures
Export **two** figures of the EV network: (1) **size = betweenness**, (2) **size = PageRank** — both coloured by Modularity Class. Which users look important in one figure but not in the other?

**Your notes (Challenge 2):** _2–3 sentences and the two file names._

## Challenge 3 · Explain the differences
Name **two reasons** why Gephi and Python can give slightly different numbers.

> 💡 **Hint:** think about **randomness** in Louvain (Gephi’s *Randomize* option vs. our `seed=42`), **resolution**, **edge weights**, **directed vs. undirected** import, and **normalisation**.

**Your interpretation (Challenge 3):** _2–3 sentences._

## Reflect
One paragraph: **when would you use Python, and when Gephi?** Think about reproducibility (code, seeds) versus exploration (seeing clusters and bridges).